# 14.7 Saving & Loading 💾🔥

The three ideas are:

```text
Model Weights
      ↓
Save what the model learned

Checkpoint
      ↓
Save the training state

Resume Training
      ↓
Load the checkpoint and continue
```

---

# 1. Model Weights 🔥🔥🔥

Remember what we learned earlier:

A neural network has **parameters**:

$$
W,\ b
$$

During training:

```text
Initial weights
      ↓
Forward
      ↓
Loss
      ↓
Backpropagation
      ↓
Optimizer
      ↓
Updated weights
      ↓
Updated weights
      ↓
...
```

The **weights and biases learned by the model** are what we usually want to save.

In PyTorch, these are available through:

```python
model.state_dict()
```

---

# 2. What is `state_dict()`?

A `state_dict` is essentially a dictionary containing the model's **learnable parameters** and relevant registered state.

For example:

```python
model.state_dict()
```

might contain things conceptually like:

```text
{
    "fc1.weight": ...,
    "fc1.bias": ...,
    "fc2.weight": ...,
    "fc2.bias": ...
}
```

So:

```text
Model
 ↓
state_dict()
 ↓
parameters/state
 ↓
save
```

PyTorch recommends saving a model's `state_dict` as the standard approach for saving learned parameters. [PyTorch — Saving and Loading Models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html?utm_source=chatgpt.com)

---

# 3. Saving Model Weights 🔥🔥🔥

Suppose:

```python
model = MyModel()
```

After training:

```python
torch.save(model.state_dict(), "model_weights.pth")
```

Now you have:

```text
model_weights.pth
```

containing the learned model parameters.

### Important

You are **not saving the entire Python model architecture here**.

You're saving its learned state.

Think:

```text
Architecture
    +
Weights
    ↓
Trained model
```

The `state_dict` contains the **weights**, while your Python code defines the **architecture**.

---

# 4. Loading Model Weights 🔥🔥🔥

Suppose you start a new Python program.

You recreate the architecture:

```python
model = MyModel()
```

Then load the saved weights:

```python
model.load_state_dict(
    torch.load("model_weights.pth")
)
```

Then:

```python
model.eval()
```

for inference.

### Complete flow

```text
TRAINING

Model architecture
       +
Learned weights
       ↓
state_dict
       ↓
model_weights.pth


LATER

Create same architecture
       ↓
Load state_dict
       ↓
Trained model
       ↓
Inference
```

---

# 5. Why Architecture Is Needed

Suppose you saved:

```text
fc1.weight
fc1.bias
fc2.weight
fc2.bias
```

PyTorch needs to know:

```text
What is fc1?
What shape should fc1.weight have?
What is fc2?
What shape should fc2.weight have?
```

That's why you recreate the model architecture first.

For example:

```python
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)
```

Then load:

```python
model.load_state_dict(
    torch.load("model_weights.pth")
)
```

---

# 6. Saving Weights for Inference 🔥🔥🔥

This is a very common workflow.

### Training

```python
model.train()

# training...
```

Then save:

```python
torch.save(
    model.state_dict(),
    "model_weights.pth"
)
```

### Later

```python
model = MyModel()

model.load_state_dict(
    torch.load("model_weights.pth")
)

model.eval()
```

Then:

```python
with torch.no_grad():
    prediction = model(X)
```

So:

```text
Training
   ↓
Learn weights
   ↓
Save state_dict
   ↓
Deploy / inference later
   ↓
Recreate architecture
   ↓
Load weights
   ↓
eval()
   ↓
Prediction
```

---

# 7. What is a Checkpoint? 🔥🔥🔥

Now comes the important difference.

### Model weights

Save:

> **What has the model learned?**

### Checkpoint

Save:

> **Where was the training process?**

A checkpoint can contain much more than model weights.

For example:

```python
checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss,
}
```

Then:

```python
torch.save(
    checkpoint,
    "checkpoint.pth"
)
```

---

# 8. Why Save the Optimizer?

This is one of the most important ideas in this topic. 🔥

Remember our optimizers from Module 12.

For Adam, the optimizer maintains additional state:

```text
Adam
├── parameters
├── first moment
└── second moment
```

Specifically:

$$
m_t
$$

and

$$
v_t
$$

These contain information about the previous gradients.

If you only save:

```python
model.state_dict()
```

you save the model's learned parameters.

But you **don't preserve the optimizer's internal state**.

Therefore:

```text
Model weights saved
        ↓
Optimizer history lost
```

If you're trying to **continue training**, that can matter.

---

# 9. Model Weights vs Checkpoint ⭐⭐⭐

|                  | Model Weights | Checkpoint |
| ---------------- | ------------- | ---------- |
| Model parameters | ✅             | ✅          |
| Optimizer state  | ❌             | ✅          |
| Epoch            | ❌             | ✅          |
| Training loss    | Usually ❌     | Can save   |
| Scheduler state  | ❌             | Can save   |
| Resume training  | Not complete  | ✅          |
| Inference        | ✅             | ✅          |

### Simple memory trick

> **Weights = what the model learned.**

> **Checkpoint = where the training process was.**

---

# 10. Saving a Checkpoint 🔥🔥🔥

Example:

```python
checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item()
}

torch.save(
    checkpoint,
    "checkpoint.pth"
)
```

Now the file contains information about the training state.

---

# 11. Loading a Checkpoint 🔥🔥🔥

First recreate the model:

```python
model = MyModel()
```

Then recreate the optimizer:

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)
```

Then load:

```python
checkpoint = torch.load("checkpoint.pth")

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)
```

Retrieve the epoch:

```python
start_epoch = checkpoint["epoch"] + 1
```

Now you can continue training.

---

# 12. Resume Training 🔥🔥🔥

Suppose you trained:

```text
Epoch 1
Epoch 2
Epoch 3
...
Epoch 20
```

Then your computer crashes. 😭

Without a checkpoint:

```text
Start again
 ↓
Epoch 1
Epoch 2
...
```

With a checkpoint:

```text
Checkpoint
   ↓
Epoch 20
   ↓
Load model
   ↓
Load optimizer
   ↓
Continue
   ↓
Epoch 21
Epoch 22
...
```

That's the purpose of checkpointing.

---

# 13. Complete Resume Example

### During training

```python
for epoch in range(num_epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        prediction = model(X_batch)

        loss = loss_fn(prediction, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss.item()
    }

    torch.save(
        checkpoint,
        "checkpoint.pth"
    )
```

This saves a checkpoint after each epoch.

---

# 14. Loading and Continuing

```python
checkpoint = torch.load(
    "checkpoint.pth",
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1
```

Then:

```python
for epoch in range(start_epoch, num_epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        prediction = model(X_batch)

        loss = loss_fn(prediction, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

The important idea is:

```text
checkpoint
   ↓
model state
   +
optimizer state
   +
epoch
   ↓
continue training
```

---

# 15. Checkpoint + Learning Rate Scheduler 🔥

Later you'll encounter **learning-rate schedulers**.

For example:

```text
Epoch 1 → LR = 0.001
Epoch 2 → LR = 0.001
...
Epoch 10 → LR = 0.0005
```

If you want to resume training properly, you may also save the scheduler state.

```python
checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "loss": loss.item()
}
```

Then:

```python
scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)
```

### General idea

A serious training checkpoint may contain:

```text
Checkpoint
│
├── Model state
├── Optimizer state
├── Scheduler state
├── Epoch
├── Loss
└── Other training metadata
```

---

# 16. Best Model vs Latest Checkpoint

These are slightly different ideas.

### Latest checkpoint

```text
Epoch 20
 ↓
save
```

Then:

```text
Epoch 21
 ↓
save
```

The latest checkpoint tells you:

> Where did training most recently stop?

### Best model

Suppose:

```text
Epoch     Validation Loss

10        0.42
11        0.38
12        0.35  ← best
13        0.37
14        0.41
```

You may want to save the model from **epoch 12**, because it had the best validation performance.

```python
if val_loss < best_val_loss:

    best_val_loss = val_loss

    torch.save(
        model.state_dict(),
        "best_model.pth"
    )
```

So you can have:

```text
best_model.pth
     ↓
best validation performance

checkpoint.pth
     ↓
latest training state
```

This distinction is extremely useful in real projects.

---

# 17. Saving Model Weights vs Checkpoint — Visual

```text
                 TRAINING
                    │
                    ▼
              Neural Network
                    │
          ┌─────────┴─────────┐
          │                   │
          ▼                   ▼
   Model Parameters       Training State
     W, b, etc.           epoch
          │               optimizer
          │               scheduler
          │               loss
          ▼                   │
    state_dict()              │
          │                   │
          ▼                   ▼
  best_model.pth        checkpoint.pth
          │                   │
          ▼                   ▼
     Inference          Resume training
```

---

# 18. Saving with GPU Models ⚠️

Suppose the model was trained on GPU:

```python
model = model.to("cuda")
```

You may later want to load the weights on a CPU-only machine.

PyTorch provides `map_location` for this.

For example:

```python
checkpoint = torch.load(
    "checkpoint.pth",
    map_location="cpu",
    weights_only=False
)
```

This tells PyTorch to load tensors onto CPU rather than requiring the original CUDA device.

For a weights-only file:

```python
state_dict = torch.load(
    "model_weights.pth",
    map_location="cpu"
)

model.load_state_dict(state_dict)
```

This is very useful when moving between:

```text
GPU training machine
        ↓
CPU inference/development machine
```

---

# 19. `.pth` vs `.pt`

You'll commonly see:

```text
model.pth
model.pt
checkpoint.pth
```

These are commonly used PyTorch file extensions.

The extension itself doesn't fundamentally determine what is stored.

For example:

```python
torch.save(model.state_dict(), "model.pth")
```

and:

```python
torch.save(model.state_dict(), "model.pt")
```

can both store a state dictionary.

So don't think:

> `.pth` = weights only

or:

> `.pt` = complete model

The **content depends on what you pass to `torch.save()`**.

---

# 20. Security / Loading Note ⚠️

When loading model files, prefer loading **state dictionaries from trusted sources**.

For modern PyTorch versions, `torch.load()` has increasingly emphasized safer weights-only loading for cases where only tensor weights are needed.

For example:

```python
state_dict = torch.load(
    "model_weights.pth",
    weights_only=True
)
```

when loading a normal state dictionary.

For full training checkpoints containing arbitrary Python objects, the exact loading requirements depend on what was stored. Only load checkpoint files you trust.

---

# 21. Industry Workflow 🔥🔥🔥

A realistic training project might look like:

```text
training/
│
├── latest_checkpoint.pth
├── best_model.pth
├── config.yaml
├── train.py
└── logs/
```

During training:

```text
Epoch
 ↓
Train
 ↓
Validation
 ↓
Save latest checkpoint
 ↓
Is validation better?
       │
      Yes
       ↓
Save best model
```

Then:

```text
Training interrupted?
        │
       Yes
        ↓
Load latest checkpoint
        ↓
Resume training
```

For deployment:

```text
best_model.pth
       ↓
Create architecture
       ↓
Load weights
       ↓
model.eval()
       ↓
Inference
```

---

# 22. Common Mistakes 🚨

### ❌ Mistake 1 — Saving only the model object and expecting training to resume

Saving only model parameters does not preserve optimizer history.

For resume training, save:

```python
model.state_dict()
optimizer.state_dict()
```

and relevant training metadata.

---

### ❌ Mistake 2 — Forgetting to recreate the architecture

You generally need the model architecture before loading its `state_dict`.

```python
model = MyModel()

model.load_state_dict(...)
```

---

### ❌ Mistake 3 — Loading weights but forgetting `eval()`

For inference:

```python
model.load_state_dict(...)
model.eval()
```

Especially important when the model contains:

* Dropout
* BatchNorm

---

### ❌ Mistake 4 — Saving only the latest model

If validation performance starts getting worse:

```text
Epoch 10 → best
Epoch 11 → worse
Epoch 12 → worse
Epoch 13 → worse
```

If you only keep epoch 13, you've lost the best state.

That's why:

```text
best_model
+
latest_checkpoint
```

is a useful pattern.

---

# 23. What You Must Know ⭐⭐⭐

## 🔥🔥🔥 MUST KNOW

You should be able to explain:

### Model weights

```python
model.state_dict()
```

### Save weights

```python
torch.save(
    model.state_dict(),
    "model.pth"
)
```

### Load weights

```python
model.load_state_dict(
    torch.load("model.pth")
)
```

### Checkpoint

A checkpoint can contain:

```text
model state
+
optimizer state
+
epoch
+
loss
+
scheduler state
```

### Resume

```text
Load checkpoint
      ↓
Load model state
      ↓
Load optimizer state
      ↓
Recover epoch
      ↓
Continue training
```

---

## 🔥🔥 IMPORTANT

Understand:

* `state_dict()`
* `optimizer.state_dict()`
* Best model vs latest checkpoint
* `map_location`
* Saving scheduler state
* Why optimizer state matters for Adam/AdamW
* `.pth` vs `.pt`

---

# 🧠 Final Cheat Sheet

| Concept                  | Meaning                                 |
| ------------------------ | --------------------------------------- |
| `model.state_dict()`     | Model's learned/registered state        |
| `optimizer.state_dict()` | Optimizer's internal state              |
| Model weights            | What the model learned                  |
| Checkpoint               | Snapshot of training state              |
| `torch.save()`           | Save object/state                       |
| `load_state_dict()`      | Load saved state into model             |
| `best_model.pth`         | Model with best validation performance  |
| `checkpoint.pth`         | Training snapshot used for continuation |
| `map_location`           | Controls where loaded tensors go        |
| Resume training          | Continue from saved training state      |

### 🔑 Best memory sentence

> **Weights tell you what the model learned; a checkpoint tells you where the training process was.**

And the overall workflow:

```text
                 TRAINING
                    │
                    ▼
              Model + Optimizer
                    │
                    ▼
                Training
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
    Best validation       Latest state
          │                   │
          ▼                   ▼
    best_model.pth       checkpoint.pth
          │                   │
          ▼                   ▼
      Inference          Resume training
```

---
